In [1]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ALEJANDRO MELGUIZO
# DATE: 8/18/2026
# TOPIC: new file to filter and clean data for analysis
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
#set working directory
os.chdir("C:/Users/A.Melguizo001/Downloads/Immig, Wages, Latinos")

#preliminary coding
    #defining data frames
df_1 = pd.read_csv('gaston_df_v2.csv')
df_hisp = df_1.copy()

## New var construction

In [3]:
#construcing a new var that measures roughly whether a person got their bachelor's or higher in the US
df_hisp['US_educ'] = 0

hispan_educ_cases = [
    ((df_hisp['educ_clean'] >= 3) & (df_hisp['AGEATIMMIG'] <= 17), 1)
]

df_hisp['US_educ'] = (
    pd.Series(np.nan, index = df_hisp.index)
    .case_when(hispan_educ_cases)
    .fillna(0)
    .astype(int)
)

In [4]:
hispan_race_cases = [
    ((df_hisp['RACE'] == 100), 0), #white
    ((df_hisp['RACE'] == 200), 1), #black
    ((df_hisp['RACE'] == 300), 2), #native american
    ((df_hisp['RACE'].isin(range(650, 653))), 3), #asian / pacific islander
    ((df_hisp['RACE'] == 700), 4),
    ((df_hisp['RACE'].isin(range(801, 831))), 4),
    ((df_hisp['RACE'] == 999), np.nan)
]

df_hisp['race_clean'] = (
    pd.Series(np.nan, index = df_hisp.index)
    .case_when(hispan_race_cases)
    .fillna(np.nan)
    .astype(int)
)

## Data Cleaning

In [5]:
#dropping hisp_clean to re-categorize
df_hisp = df_hisp.drop(columns= ['hisp_clean'])

#re-categorizing hispanic category
hispan_cases = [
    (df_hisp['HISPAN'] == 0, 0),
    ((df_hisp['HISPAN'] >= 100) & (df_hisp['HISPAN'] <= 109), 1),
    (df_hisp['HISPAN'] == 200, 2),
    (df_hisp['HISPAN'] == 300, 3), 
    (df_hisp['HISPAN'] == 400, 4),
    (df_hisp['HISPAN'] == 500, 5),
    ((df_hisp['HISPAN'] == 610) | (df_hisp['HISPAN'] == 611), 6),
    (df_hisp['HISPAN'] == 612, 7),
    (df_hisp['HISPAN'] == 600, 8),
    ((df_hisp['HISPAN'].isna()) | (df_hisp['HISPAN'] == 902), 9)
]

df_hisp['hisp_clean'] = (
    pd.Series(np.nan, index = df_hisp.index)
    .case_when(hispan_cases)
    .fillna(9)
    .astype(int)
)

In [8]:
#filter to only include:

#respondents who are: hispanic, born outside of the US and territories, have data for AGEATIMMIG, and have a non-0 wage.
df_hisp = df_hisp[
    (df_hisp['hisp_clean'] > 0) &
    (df_hisp['bpl_binary'] == 1) &
    (df_hisp['AGEATIMMIG'].notna()) &
    (df_hisp['incwage_clean'] > 0)
]

#making log wages for regression
df_hisp['log_wage'] = np.log(df_hisp['incwage_clean'])

## non-linear regressors

In [10]:
df_hisp['AAI_sq'] = df_hisp['AGEATIMMIG'] * df_hisp['AGEATIMMIG']

In [11]:

# Massachusetts filter
df_hisp_MA = df_hisp[
    (df_hisp['STATEFIP'] == 25)
]

## Regression modeling using statsmodels

### MA Hispanic Regression

In [ ]:
#full regression on MA hispanic data
# model_MA = smf.ols(formula = 'log_wage ~ years_since_immig + C(hisp_clean) + sex_clean + C(educ_clean) + C(race_clean) + UHRSWORKT + C(EMPSTAT) + C(STATEFIP)', data = df_hisp_MA)
# results_MA = model_MA.fit()

# print(results_MA.summary())

### Overall US Hispanic regression

In [16]:
#regression on US hispanic data
model = smf.ols(formula = """
log_wage ~ 
AGE +
years_since_immig + 
sex_clean + 
UHRSWORKT + 
US_educ +
sex_clean * C(educ_clean) + 
sex_clean * C(hisp_clean) +
C(CITIZEN) +
C(hisp_clean) + 
C(educ_clean) + 
C(race_clean) + 
C(EMPSTAT) + 
C(STATEFIP)
"""
, data = df_hisp)
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:               log_wage   R-squared:                       0.246
Model:                            OLS   Adj. R-squared:                  0.246
Method:                 Least Squares   F-statistic:                     579.1
Date:                Tue, 25 Aug 2026   Prob (F-statistic):               0.00
Time:                        12:14:48   Log-Likelihood:            -1.8682e+05
No. Observations:              165152   AIC:                         3.738e+05
Df Residuals:                  165058   BIC:                         3.748e+05
Df Model:                          93                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       